# Apex Retail Intelligence — Phase 1 & 2: Raw Ingestion + Landing Zone

**Deliverable:** Raw & Landing Script

This notebook:
1. Reads incoming CSVs (historical + incremental) for `customer`, `product`, `sales` from the inbound Volume.
2. Organizes them into a `raw/` directory structure (all columns as STRING).
3. Converts Raw CSV -> Parquet (Landing zone), preserving STRING types.
4. Dynamically reads `audit_landing_*.csv` files and validates actual row counts against declared
   expected counts, producing a PASS/FAIL report. The pipeline halts (raises) on any FAIL.

Idempotency: every write uses `mode("overwrite")` per (dataset, load_type) partition-of-work, so re-running
this notebook never duplicates raw/landing data — it simply re-materializes it.

## 0. Configuration (Widgets)
Change these to match your environment. Keeping them as widgets means the logic below never
has to be touched when paths change — a key requirement for a maintainable, reusable pipeline.

In [0]:
dbutils.widgets.text("inbound_volume", "/Volumes/apex_retail/landing/inbound_files", "Inbound Volume Path")
dbutils.widgets.text("raw_base_path", "/Volumes/apex_retail/landing/raw", "Raw Zone Base Path")
dbutils.widgets.text("landing_base_path", "/Volumes/apex_retail/landing/parquet", "Landing (Parquet) Base Path")




INBOUND_PATH = dbutils.widgets.get("inbound_volume")
RAW_BASE = dbutils.widgets.get("raw_base_path")
LANDING_BASE = dbutils.widgets.get("landing_base_path")

# The three core datasets and the two load types the assignment defines.
DATASETS = ["customer", "product", "sales"]
LOAD_TYPES = ["historical", "incremental"]

print(f"Inbound volume : {INBOUND_PATH}")
print(f"Raw base path  : {RAW_BASE}")
print(f"Landing base   : {LANDING_BASE}")

Inbound volume : /Volumes/apex_retail/landing/inbound_files
Raw base path  : /Volumes/apex_retail/landing/raw
Landing base   : /Volumes/apex_retail/landing/parquet


## 1. Phase 1 — Raw Extraction
Read every incoming CSV **as STRING** (no type inference) and write into a `raw/<load_type>/<dataset>/`
directory structure. Reading everything as string at this stage is intentional: Raw is a faithful,
lossless copy of what arrived — type casting happens later, in Silver, where we control it explicitly.

In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.types import StructType

# Expected filename convention in the inbound volume:
#   <dataset>_historical.csv   e.g. customer_historical.csv
#   <dataset>_incremental.csv  e.g. sales_incremental.csv
# Adjust FILENAME_PATTERN if your actual filenames differ.

def build_inbound_filename(dataset: str, load_type: str) -> str:
    return f"{dataset}_{load_type}.csv"


def read_raw_csv_as_string(file_path: str) -> DataFrame:
    """Read a CSV with every column forced to StringType — no schema inference,
    no type coercion. This guarantees Raw is a byte-faithful representation of source data."""
    # Read once to discover column names only (header row), then rebuild an all-string schema.
    header_df = spark.read.option("header", "true").csv(file_path)
    string_schema = StructType([field.withColumnRenamed if False else field for field in header_df.schema.fields])
    # Force every field to string regardless of what CSV inference guessed.
    from pyspark.sql.types import StringType, StructField
    all_string_schema = StructType(
        [StructField(f.name, StringType(), True) for f in header_df.schema.fields]
    )
    return (
        spark.read
        .option("header", "true")
        .option("multiLine", "true")
        .option("escape", '"')
        .schema(all_string_schema)
        .csv(file_path)
    )


raw_row_counts = {}  # (dataset, load_type) -> row count, used later for a sanity cross-check

for dataset in DATASETS:
    for load_type in LOAD_TYPES:
        src_file = f"{INBOUND_PATH}/{build_inbound_filename(dataset, load_type)}"
        dest_dir = f"{RAW_BASE}/{load_type}/{dataset}"

        print(f"[RAW] Reading {src_file} ...")
        df_raw = read_raw_csv_as_string(src_file)

        row_count = df_raw.count()
        raw_row_counts[(dataset, load_type)] = row_count

        # overwrite -> idempotent: re-running this cell never appends duplicate raw copies
        (
            df_raw.write
            .mode("overwrite")
            .option("header", "true")
            .csv(dest_dir)
        )
        print(f"[RAW] Wrote {row_count} rows -> {dest_dir}")

[RAW] Reading /Volumes/apex_retail/landing/inbound_files/customer_historical.csv ...
[RAW] Wrote 50 rows -> /Volumes/apex_retail/landing/raw/historical/customer
[RAW] Reading /Volumes/apex_retail/landing/inbound_files/customer_incremental.csv ...
[RAW] Wrote 10 rows -> /Volumes/apex_retail/landing/raw/incremental/customer
[RAW] Reading /Volumes/apex_retail/landing/inbound_files/product_historical.csv ...
[RAW] Wrote 30 rows -> /Volumes/apex_retail/landing/raw/historical/product
[RAW] Reading /Volumes/apex_retail/landing/inbound_files/product_incremental.csv ...
[RAW] Wrote 5 rows -> /Volumes/apex_retail/landing/raw/incremental/product
[RAW] Reading /Volumes/apex_retail/landing/inbound_files/sales_historical.csv ...
[RAW] Wrote 100 rows -> /Volumes/apex_retail/landing/raw/historical/sales
[RAW] Reading /Volumes/apex_retail/landing/inbound_files/sales_incremental.csv ...
[RAW] Wrote 20 rows -> /Volumes/apex_retail/landing/raw/incremental/sales


## 2. Phase 2 — Landing Conversion (CSV -> Parquet)
Convert the Raw (string) CSVs into Parquet. Parquet's columnar layout + compression makes every
downstream Spark read (Bronze onward) dramatically faster and cheaper than re-reading CSV each time.
Types remain STRING here — Landing is still a "shape change", not a "meaning change".

In [0]:
landing_row_counts = {}

for dataset in DATASETS:
    for load_type in LOAD_TYPES:
        src_dir = f"{RAW_BASE}/{load_type}/{dataset}"
        dest_dir = f"{LANDING_BASE}/{load_type}/{dataset}"

        df_raw = spark.read.option("header", "true").csv(src_dir)  # already string-typed source
        row_count = df_raw.count()
        landing_row_counts[(dataset, load_type)] = row_count

        (
            df_raw.write
            .mode("overwrite")               # idempotent re-run
            .parquet(dest_dir)
        )
        print(f"[LANDING] {dataset}/{load_type}: {row_count} rows -> {dest_dir} (parquet)")

[LANDING] customer/historical: 50 rows -> /Volumes/apex_retail/landing/parquet/historical/customer (parquet)
[LANDING] customer/incremental: 10 rows -> /Volumes/apex_retail/landing/parquet/incremental/customer (parquet)
[LANDING] product/historical: 30 rows -> /Volumes/apex_retail/landing/parquet/historical/product (parquet)
[LANDING] product/incremental: 5 rows -> /Volumes/apex_retail/landing/parquet/incremental/product (parquet)
[LANDING] sales/historical: 100 rows -> /Volumes/apex_retail/landing/parquet/historical/sales (parquet)
[LANDING] sales/incremental: 20 rows -> /Volumes/apex_retail/landing/parquet/incremental/sales (parquet)


## 3. Data Auditing — Dynamic Audit File Validation
For every `audit_landing_<dataset>_<load_type>.csv` (or however your audit files are named — see
`AUDIT_FILENAME_PATTERN`), read the declared `expected_row_count` for `table_name` and compare it
against what actually landed. A single mismatch halts the whole pipeline with a raised exception,
per the assignment's hard requirement.

In [0]:
dbutils.widgets.text("audit_path", "/Volumes/apex_retail/landing/audit", "Audit Files Path")
AUDIT_PATH = dbutils.widgets.get("audit_path")

def build_audit_filename(dataset: str, load_type: str) -> str:
    return f"audit_landing_{dataset}_{load_type}.csv"


audit_results = []  # collects rows for the structured PASS/FAIL report
pipeline_failed = False

for dataset in DATASETS:
    for load_type in LOAD_TYPES:
        audit_file = f"{AUDIT_PATH}/{build_audit_filename(dataset, load_type)}"

        try:
            audit_df = spark.read.option("header", "true").option("inferSchema", "true").csv(audit_file)
            audit_row = audit_df.collect()[0]  # one row per audit file, per assignment spec
            expected_count = int(audit_row["row_count"])
        except Exception as e:
            audit_results.append({
                "table_name": dataset, "load_type": load_type,
                "expected": None, "actual": landing_row_counts.get((dataset, load_type)),
                "status": "FAIL", "reason": f"Could not read audit file: {e}"
            })
            pipeline_failed = True
            continue

        actual_count = landing_row_counts.get((dataset, load_type))
        status = "PASS" if actual_count == expected_count else "FAIL"
        if status == "FAIL":
            pipeline_failed = True

        audit_results.append({
            "table_name": dataset, "load_type": load_type,
            "expected": expected_count, "actual": actual_count,
            "status": status, "reason": "" if status == "PASS" else "Row count mismatch"
        })

### Structured PASS/FAIL Report

In [0]:
from pyspark.sql.functions import lit

audit_report_df = spark.createDataFrame(audit_results)
audit_report_df = audit_report_df.withColumn("sample_number", lit(50))
display(audit_report_df)  # display() shows all records in a rich table, not just 6

# Persist the audit report itself for traceability (auditable requirement from the assignment).
(
    audit_report_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .format("delta")
    .save(f"{LANDING_BASE}/_audit_reports/landing_audit_report")
)

if pipeline_failed:
    raise Exception(
        "Landing-layer audit validation FAILED. See audit_report_df above for details. "
        "Pipeline halted per requirement — Bronze layer will not run until this is resolved."
    )
else:
    print("All Landing-layer audit checks PASSED. Safe to proceed to Bronze layer.")

actual,expected,load_type,reason,status,table_name,sample_number
50,50,historical,,PASS,customer,50
10,10,incremental,,PASS,customer,50
30,30,historical,,PASS,product,50
5,5,incremental,,PASS,product,50
100,100,historical,,PASS,sales,50
20,20,incremental,,PASS,sales,50


All Landing-layer audit checks PASSED. Safe to proceed to Bronze layer.
